# 04 结果汇总与报告导出


本笔记本在重建清洗样本并重新估计模型后，完成结果汇总、核心讨论输出与 README 生成。


## 第四部分：结果汇总与报告


In [1]:
import os
import re
from io import StringIO

import numpy as np
import pandas as pd

OUTPUT_PATH = 'output'
os.makedirs(OUTPUT_PATH, exist_ok=True)


def load_model_table(model_path):
    with open(model_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Protect header token that contains pipes.
    text = text.replace('Pr(>|t|)', 'Pr_gt_t')

    lines = [ln for ln in text.splitlines() if ln.strip().startswith('|')]
    if len(lines) < 3:
        return pd.DataFrame()

    # Keep only real table rows, skip markdown alignment row only.
    body = []
    align_pat = re.compile(r'^\|\s*:?-{2,}.*\|\s*$')
    for ln in lines:
        if align_pat.match(ln.strip()):
            continue
        body.append(ln)

    if len(body) < 2:
        return pd.DataFrame()

    table_text = '\n'.join(body)
    df = pd.read_csv(StringIO(table_text), sep='|', engine='python')
    df = df.drop(columns=[col for col in df.columns if str(col).strip() == '' or str(col).startswith('Unnamed')], errors='ignore')
    df.columns = [str(c).strip() for c in df.columns]

    # numeric conversion where possible
    for col in ['Estimate', 'Std. Error', 't value', 'Pr_gt_t', '2.5%', '97.5%']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    if 'Coefficient' in df.columns:
        df['Coefficient'] = df['Coefficient'].astype(str).str.strip()

    return df


def get_metric_from_model(model_df, coef_name, metric='Estimate'):
    if model_df.empty or 'Coefficient' not in model_df.columns or metric not in model_df.columns:
        return np.nan
    row = model_df.loc[model_df['Coefficient'] == coef_name]
    if row.empty:
        return np.nan
    return row.iloc[0][metric]


model_files = {
    'M1: TWFE': 'model_m1.txt',
    "M1': IFE": 'model_m1_ife.txt',
    'M2a: SOE': 'model_m2_soe.txt',
    'M2b: Non-SOE': 'model_m2_nonsoe.txt',
    'M3: Interaction': 'model_m3.txt',
}

model_tables = {}
for model_name, fn in model_files.items():
    path = os.path.join(OUTPUT_PATH, fn)
    model_tables[model_name] = load_model_table(path) if os.path.exists(path) else pd.DataFrame()


def fmt_coef(df_model, var):
    est = get_metric_from_model(df_model, var, 'Estimate')
    se = get_metric_from_model(df_model, var, 'Std. Error')
    p = get_metric_from_model(df_model, var, 'Pr_gt_t')
    if pd.isna(est) or pd.isna(se):
        return ''
    stars = '***' if (not pd.isna(p) and p < 0.01) else ('**' if (not pd.isna(p) and p < 0.05) else ('*' if (not pd.isna(p) and p < 0.1) else ''))
    return f"{est:.4f}{stars} ({se:.4f})"


summary_rows = []
for model_name in ['M1: TWFE', "M1': IFE", 'M2a: SOE', 'M2b: Non-SOE', 'M3: Interaction']:
    tbl = model_tables.get(model_name, pd.DataFrame())
    summary_rows.append({
        'Model': model_name,
        'NPR': fmt_coef(tbl, 'NPR'),
        'NPR x SOE': fmt_coef(tbl, 'NPR_SOE'),
        'm2_growth': fmt_coef(tbl, 'm2_growth'),
        'Size': fmt_coef(tbl, 'Size'),
        'Tang': fmt_coef(tbl, 'Tang'),
        'Growth': fmt_coef(tbl, 'Growth'),
        'NDTS': fmt_coef(tbl, 'NDTS'),
        'Firm FE': 'Yes',
        'Year FE': 'Yes' if model_name != "M1': IFE" else 'Yes (with m2_growth)',
        'SE clustering': 'Two-way (firm, year)'
    })

regression_summary = pd.DataFrame(summary_rows)

try:
    regression_summary.to_excel(os.path.join(OUTPUT_PATH, 'regression_summary.xlsx'), index=False)
    print('Saved: output/regression_summary.xlsx')
except Exception as e:
    print(f'Excel export skipped: {e}')

regression_summary.to_csv(os.path.join(OUTPUT_PATH, 'regression_summary.csv'), index=False, encoding='utf-8-sig')
print('Saved: output/regression_summary.csv')
print('Regression summary table (M1-M3) generated from output/model_*.txt')
print(regression_summary)



Saved: output/regression_summary.xlsx
Saved: output/regression_summary.csv
Regression summary table (M1-M3) generated from output/model_*.txt
             Model                  NPR            NPR x SOE  \
0         M1: TWFE  -0.4730*** (0.0390)                        
1         M1': IFE  -0.4730*** (0.0390)                        
2         M2a: SOE  -0.6430*** (0.0480)                        
3     M2b: Non-SOE  -0.3610*** (0.0360)                        
4  M3: Interaction  -0.4110*** (0.0390)  -0.2330*** (0.0530)   

              m2_growth                Size                Tang  \
0                        0.0800*** (0.0040)  0.2350*** (0.0140)   
1  -0.9510 (10454.3910)  0.0800*** (0.0040)  0.2350*** (0.0140)   
2                        0.0970*** (0.0080)  0.1620*** (0.0180)   
3                        0.0670*** (0.0050)  0.2640*** (0.0200)   
4                        0.0800*** (0.0040)  0.2340*** (0.0140)   

               Growth                 NDTS Firm FE               Year 

### 4.2 核心讨论问题


In [2]:
print('Discussion summary:')
print('=' * 80)

# 1) Theory check using M1
m1_tbl = model_tables.get('M1: TWFE', pd.DataFrame())
beta_m1 = get_metric_from_model(m1_tbl, 'NPR', 'Estimate')
p_m1 = get_metric_from_model(m1_tbl, 'NPR', 'Pr_gt_t')

if pd.notna(beta_m1):
    significance = 'significant' if (pd.notna(p_m1) and p_m1 < 0.05) else 'not significant'
    theory = 'Pecking Order Theory' if beta_m1 < 0 else 'Trade-off Theory'
    print('1. Theory test:')
    print(f'   - M1 NPR coefficient: {beta_m1:.4f}')
    print(f'   - p-value: {p_m1:.4f} ({significance})' if pd.notna(p_m1) else '   - p-value: missing')
    print(f'   - Implied support: {theory}')
else:
    print('1. Theory test: failed to read NPR coefficient from model_m1.txt.')

# 2) Group difference (M2 SOE vs Non-SOE)
m2_soe_tbl = model_tables.get('M2a: SOE', pd.DataFrame())
m2_nonsoe_tbl = model_tables.get('M2b: Non-SOE', pd.DataFrame())
beta_soe = get_metric_from_model(m2_soe_tbl, 'NPR', 'Estimate')
beta_nonsoe = get_metric_from_model(m2_nonsoe_tbl, 'NPR', 'Estimate')

print('\n2. Group heterogeneity:')
if pd.notna(beta_soe) and pd.notna(beta_nonsoe):
    print(f'   - SOE: {beta_soe:.4f}')
    print(f'   - Non-SOE: {beta_nonsoe:.4f}')
    print(f'   - Difference (SOE - Non-SOE): {beta_soe - beta_nonsoe:.4f}')
else:
    print('   - failed to read NPR coefficients from model_m2_soe.txt / model_m2_nonsoe.txt.')

# 3) Time-varying coefficients (M4)
m4_path = os.path.join(OUTPUT_PATH, 'model_m4.txt')
print('\n3. Time-varying coefficients:')
if os.path.exists(m4_path):
    m4_tbl = load_model_table(m4_path)
    if not m4_tbl.empty and 'Coefficient' in m4_tbl.columns and 'Pr_gt_t' in m4_tbl.columns:
        yr = m4_tbl[m4_tbl['Coefficient'].astype(str).str.startswith('NPR_year_')].copy()
        if not yr.empty:
            yr['year'] = yr['Coefficient'].str.replace('NPR_year_', '', regex=False)
            sig = yr[(pd.notna(yr['Pr_gt_t'])) & (yr['Pr_gt_t'] < 0.05)]['year'].tolist()
            nsig = yr[(pd.notna(yr['Pr_gt_t'])) & (yr['Pr_gt_t'] >= 0.05)]['year'].tolist()
            print(f'   - Significant years: {sig}')
            print(f'   - Non-significant years: {nsig}')
        else:
            print('   - no NPR_year_* rows found.')
    else:
        print('   - failed to parse model_m4.txt.')
else:
    print('   - missing output/model_m4.txt')

# 4) Size regime effect (M6)
m6_small_tbl = load_model_table(os.path.join(OUTPUT_PATH, 'model_m6_small.txt')) if os.path.exists(os.path.join(OUTPUT_PATH, 'model_m6_small.txt')) else pd.DataFrame()
m6_large_tbl = load_model_table(os.path.join(OUTPUT_PATH, 'model_m6_large.txt')) if os.path.exists(os.path.join(OUTPUT_PATH, 'model_m6_large.txt')) else pd.DataFrame()
beta_small = get_metric_from_model(m6_small_tbl, 'NPR', 'Estimate')
beta_large = get_metric_from_model(m6_large_tbl, 'NPR', 'Estimate')

print('\n4. Size effect:')
if pd.notna(beta_small) and pd.notna(beta_large):
    print(f'   - Small-size NPR coefficient: {beta_small:.4f}')
    print(f'   - Large-size NPR coefficient: {beta_large:.4f}')
    print(f'   - Difference (Small - Large): {beta_small - beta_large:.4f}')
else:
    print('   - failed to read NPR coefficients from model_m6_small.txt / model_m6_large.txt.')

print('\n' + '=' * 80)
print('Summary completed.')



Discussion summary:
1. Theory test:
   - M1 NPR coefficient: -0.4730
   - p-value: 0.0000 (significant)
   - Implied support: Pecking Order Theory

2. Group heterogeneity:
   - SOE: -0.6430
   - Non-SOE: -0.3610
   - Difference (SOE - Non-SOE): -0.2820

3. Time-varying coefficients:
   - Significant years: ['2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
   - Non-significant years: ['2010']

4. Size effect:
   - Small-size NPR coefficient: -0.4140
   - Large-size NPR coefficient: -0.4920
   - Difference (Small - Large): 0.0780

Summary completed.


### 4.3 创建 README 文件


In [3]:
# Build README from output artifacts only (no dependency on df/fit objects)
sample_table_path = os.path.join(OUTPUT_PATH, 'sample_screening_table.csv')
if os.path.exists(sample_table_path):
    sample_info = pd.read_csv(sample_table_path)
    sample_table_text = sample_info.to_string(index=False)
else:
    sample_info = pd.DataFrame()
    sample_table_text = 'sample_screening_table.csv not found.'

readme_content = f"""# Capital Structure Determinants of Chinese Listed Firms

> [Assignment brief](https://github.com/lianxhcn/dsfin/blob/main/homework/ex_P03_Panel-capital_strucuture.md)

## Personal Info
- Name: Freya
- Email: [your_email@example.com]

## Data Source
- CSMAR, download date: 2026-04-20
- Use output/sample_screening_table.csv and model outputs as the latest sample reference.

## Sample Screening Table
```
{sample_table_text}
```

## Tools
- Python 3.8+
- Jupyter Notebook
- Libraries: pandas, numpy, matplotlib, seaborn, pyfixest, scipy

## GitHub Repository
https://github.com/[your-username]/dshw--panel

## Quarto Book (optional)
https://[your-username].github.io/dshw--panel/

## Main Findings (update with final results)
1. Main theoretical direction supported by M1-M3.
2. Ownership heterogeneity between SOE and Non-SOE.
3. Time-varying patterns from M4.
4. Size heterogeneity and threshold effect from M5-M6.

## File Structure
- 01_data_clean.ipynb: cleaning, variable construction, screening, winsorization
- 02_descriptive_statistics.ipynb: descriptive stats, correlation matrix, trend charts
- 03_model_estimation.ipynb: M1-M6 estimation and figures
- 04_results_summary.ipynb: summary tables, discussion, README generation
- output/
  - figures/
  - sample_screening_table.csv
  - descriptive_statistics.xlsx
  - correlation_matrix.xlsx
  - regression_summary.xlsx / regression_summary.csv
  - model_*.txt
"""

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print('README.md generated from output artifacts.')



README.md generated from output artifacts.
